# Scraping de YouTube para moderacion de contenido

**Politica y farandula peruana**

**Objetivo:** recolectar metadatos y transcripciones publicas sin usar APIs ni llaves externas. El cuaderno sigue el estilo de los ejemplos de clase: instalacion, exploracion, scraping, organizacion en tablas y almacenamiento local.

**Flujo:**
1. Definir canales semilla.
2. Buscar canales candidatos con Selenium sobre la web publica.
3. Extraer videos recientes con `yt-dlp` sin descargar video.
4. Descargar subtitulos/transcripciones publicas con `yt-dlp`.
5. Guardar un archivo `jsonl` compatible con el cuaderno de limpieza.

## 1. Instalacion de librerias

Estas librerias se usan de manera local. No se usa YouTube Data API, Google API ni servicios de LLM.

In [1]:
# Instalación de librerías del proyecto
# youtube-transcript-api es un scraper sin API key (fallback para transcripciones)
!pip install -q pandas selenium beautifulsoup4 webdriver-manager yt-dlp tqdm youtube-transcript-api


## 2. Configuracion central del scraping

Todos los parametros editables del flujo se concentran en la celda siguiente:

- **Descubrimiento de canales:** activa o desactiva Selenium y controla el modo sin ventana, las esperas y el numero de desplazamientos de la pagina.
- **Cobertura del corpus:** limita cuantos videos recientes se inspeccionan por canal.
- **Red y reintentos:** define pausas, reintentos, encabezado del navegador y validacion TLS para las consultas de `yt-dlp`.
- **Subtitulos:** establece idiomas prioritarios, fuentes permitidas (manuales, automaticos y fallback), formato y reutilizacion de archivos integros.
- **Calidad minima:** determina cuantos caracteres utiles debe tener una transcripcion para entrar al JSONL que consume el cuaderno 02.

> La omision de audio y video no es configurable: `skip_download=True` permanece como salvaguarda obligatoria en todo el cuaderno.

In [2]:
from pathlib import Path
import hashlib
import json
import math
import re
import time
from urllib.parse import quote_plus

from bs4 import BeautifulSoup
import pandas as pd
from tqdm.auto import tqdm

# ══════════════════════════════════════════════════════════════════════════════
# PARAMETROS DEL FLUJO: editar únicamente este bloque
# ══════════════════════════════════════════════════════════════════════════════

# 1) Descubrimiento opcional de canales con Selenium
EJECUTAR_BUSQUEDA_CANALES = False # False: usar directamente canales_semilla
SELENIUM_HEADLESS = True          # True: Chrome se ejecuta sin abrir ventana
SELENIUM_N_SCROLLS = 3            # desplazamientos para cargar más resultados
SELENIUM_WAIT_SECONDS = 10        # espera máxima del primer resultado
SELENIUM_FALLBACK_SECONDS = 4     # espera adicional si no aparece el selector
SELENIUM_SCROLL_PAUSE_SECONDS = 2 # pausa entre desplazamientos
NAVEGADOR_IDIOMA = 'es-PE'
NAVEGADOR_VENTANA = '1400,1000'
USER_AGENT = (
    'Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
    'AppleWebKit/537.36 (KHTML, like Gecko) '
    'Chrome/124.0.0.0 Safari/537.36'
)
CONSULTAS_BUSQUEDA = [
    # Política y periodismo
    'noticias politica peru canal youtube',
    'periodismo opinion peru youtube canal',
    'RPP Exitosa Canal N ATV Latina Peru noticias',
    'Marco Sifuentes Ocram La Encerrona YouTube Peru',
    'Curwen diario YouTube Peru canal opinion',
    'Willax Phillip Butters Beto Ortiz YouTube Peru opinion',
    'analisis politico peruano youtube podcast',
    # Humor y streaming
    'Hablando Huevadas YouTube Peru canal oficial humor',
    'Goblinciano YouTube Peru canal streaming',
    'El Cacas youtuber peruano canal oficial',
    'Negro Fuertes comedia peruana YouTube canal',
    'Jason Qqq YouTube Peru canal reacciones',
    'Todo Good streaming Peru youtube canal',
    'youtubers peruanos humor adulto lenguaje coloquial',
    'comedia peru youtube canal popular 2024',
    # Farándula y espectáculos
    'Magaly Amor y Fuego America Hoy YouTube Peru farandula',
    'Instarandula Samuel Suarez YouTube canal oficial',
    'El Popular farandula Peru youtube canal',
    'chismes espectaculos peru youtube canal 2024',
    # Deportes, viajes y cultura popular
    'Nico Moschella futbol peru YouTube canal',
    'Libero Depor futbol peruano YouTube canal comentarios',
    'comentaristas deportivos peruanos youtube canal informal',
    'youtubers peruanos viajes gastronomia Peru canal',
    'Misias Buen Viaje Viaja y Prueba YouTube Peru',
    'vloggers Peru 2024 youtube canal lifestyle',
    'humor regional peruano youtube canal serrano costeno',
    'cultura popular peruana youtube canal entretenimiento',
]

# 2) Cobertura: metadatos inspeccionados por cada canal semilla
MAX_VIDEOS_POR_CANAL = 75

# 3) Red y comportamiento de yt-dlp
YT_QUIET = True               # reduce mensajes informativos de yt-dlp
YT_NO_WARNINGS = True         # oculta advertencias no críticas
YT_IGNORE_ERRORS = True       # continúa con el siguiente video si uno falla
YT_RETRIES = 3                # reintentos ante errores temporales de red
YT_SLEEP_MIN_SECONDS = 1      # pausa mínima entre solicitudes
YT_SLEEP_MAX_SECONDS = 3      # pausa máxima aleatoria entre solicitudes
PERMITIR_SSL_INSEGURO = False  # True solo ante un proxy corporativo

# 4) Obtención y cache de subtítulos (nunca audio/video)
IDIOMAS_SUBTITULOS = ['es-PE', 'es-419', 'es']  # orden de prioridad
FORMATO_SUBTITULOS = 'vtt'    # contrato del parser; no cambiar
OBTENER_SUBTITULOS_MANUALES = True     # prioriza subtítulos del autor
OBTENER_SUBTITULOS_AUTOMATICOS = True  # respaldo generado por YouTube
USAR_TRANSCRIPT_API_FALLBACK = True     # último intento si yt-dlp no produce VTT
REUTILIZAR_CACHE = True  # omite registros/VTT tras validar su integridad
MIN_TRANSCRIPT_CHARS = 200  # umbral para aceptar y conservar una transcripción


def encontrar_raiz(inicio=Path.cwd()):
    """Encuentra la raíz del repositorio desde Jupyter o desde Cuadernos/."""
    for candidato in [inicio.resolve(), *inicio.resolve().parents]:
        if (candidato / 'Cuadernos').is_dir() and (candidato / 'datos').is_dir():
            return candidato
    raise FileNotFoundError('No se encontró la raíz del proyecto (carpetas Cuadernos/ y datos/).')


ROOT = encontrar_raiz()
RAW_DIR = ROOT / 'datos' / 'raw'
RAW_DIR.mkdir(parents=True, exist_ok=True)

if FORMATO_SUBTITULOS != 'vtt':
    raise ValueError('FORMATO_SUBTITULOS debe ser vtt porque el parser usa WebVTT.')
if MAX_VIDEOS_POR_CANAL < 1 or MIN_TRANSCRIPT_CHARS < 1:
    raise ValueError('MAX_VIDEOS_POR_CANAL y MIN_TRANSCRIPT_CHARS deben ser positivos.')
if YT_SLEEP_MIN_SECONDS > YT_SLEEP_MAX_SECONDS:
    raise ValueError('YT_SLEEP_MIN_SECONDS no puede superar YT_SLEEP_MAX_SECONDS.')
if not any([OBTENER_SUBTITULOS_MANUALES, OBTENER_SUBTITULOS_AUTOMATICOS,
            USAR_TRANSCRIPT_API_FALLBACK]):
    raise ValueError('Debe habilitarse al menos una fuente de subtítulos.')

print('Directorio de trabajo:', ROOT)
print('Salida raw:', RAW_DIR)
print('Buscar canales con Selenium:', EJECUTAR_BUSQUEDA_CANALES)
print('Máximo de videos por canal:', MAX_VIDEOS_POR_CANAL)
print('Idiomas de subtítulos:', IDIOMAS_SUBTITULOS)
print('Reutilizar cache íntegra:', REUTILIZAR_CACHE)
print('Validación TLS:', 'DESACTIVADA' if PERMITIR_SSL_INSEGURO else 'activada')


Directorio de trabajo: D:\trabajo_PLN\Trabajo_PLN-MIA-Grupo4
Salida raw: D:\trabajo_PLN\Trabajo_PLN-MIA-Grupo4\datos\raw
Buscar canales con Selenium: False
Máximo de videos por canal: 75
Idiomas de subtítulos: ['es-PE', 'es-419', 'es']
Reutilizar cache íntegra: True
Validación TLS: activada


## 3. Canales semilla

La lista inicial no es un ranking cerrado. Incluye semillas preexistentes y canales aprobados de la ultima corrida de scraping. Antes de exportarse se validan campos obligatorios, registro linguistico, URLs unicas, procedencia y evidencia de subtitulos; cada categoria debe conservar al menos dos fuentes.

In [3]:
# registro_linguistico:
#   formal      → lenguaje periodístico o académico estándar
#   informal    → coloquial pero inteligible; mezcla registros
#   coloquial   → jerga, peruanismos, humor, pace rápida
#   incorrecto  → groserías, habla popular, errores gramaticales intencionados

canales_semilla = pd.DataFrame([
    # ── Política / periodismo ─────────────────────────────────────────────────
    {'nombre': 'Marco Sifuentes / Ocram',          'categoria': 'politica_analisis',    'registro_linguistico': 'informal',   'url': 'https://www.youtube.com/@canalYAAAAA',                   'nota': 'La Encerrona; análisis político con tono irónico'},
    {'nombre': 'El diario de Curwen',              'categoria': 'politica_opinion',     'registro_linguistico': 'coloquial',  'url': 'https://www.youtube.com/@curwen',                         'nota': 'opinión política con sarcasmo y lenguaje muy coloquial peruano'},
    {'nombre': 'Sin Guion con Rosa Maria Palacios','categoria': 'politica_periodismo',  'registro_linguistico': 'formal',     'url': 'https://www.youtube.com/@singuionlr',                     'nota': 'periodismo de opinión y entrevistas'},
    {'nombre': 'RPP Noticias',                     'categoria': 'politica_actualidad',  'registro_linguistico': 'formal',     'url': 'https://www.youtube.com/@RPPNoticias',                    'nota': 'noticias y entrevistas; lenguaje periodístico estándar'},
    {'nombre': 'Exitosa Noticias',                 'categoria': 'politica_actualidad',  'registro_linguistico': 'informal',   'url': 'https://www.youtube.com/@exitosape',                      'nota': 'noticias, entrevistas y opinión; tono más popular que RPP'},
    {'nombre': 'Willax Television',                'categoria': 'politica_opinion',     'registro_linguistico': 'informal',   'url': 'https://www.youtube.com/@WillaxTV',                       'nota': 'programas de opinión política; lenguaje a veces confrontacional'},
    {'nombre': 'Canal N',                          'categoria': 'politica_actualidad',  'registro_linguistico': 'formal',     'url': 'https://www.youtube.com/@canaln',                         'nota': 'noticias y entrevistas; canal de cable informativo'},
    {'nombre': 'ATV Noticias',                     'categoria': 'politica_actualidad',  'registro_linguistico': 'informal',   'url': 'https://www.youtube.com/@ATVNoticias',                    'nota': 'noticias nacionales; lenguaje más popular que canal N'},
    {'nombre': 'Latina Noticias',                  'categoria': 'politica_actualidad',  'registro_linguistico': 'informal',   'url': 'https://www.youtube.com/@latinanoticias',                 'nota': 'noticias y magazine; amplia cobertura nacional'},
    {'nombre': 'Panamericana Noticias',            'categoria': 'politica_actualidad',  'registro_linguistico': 'informal',   'url': 'https://www.youtube.com/@Panamericana-Noticias',          'nota': 'canal informativo oficial de Panamericana Televisión'},

    # ── Humor / streaming / lenguaje coloquial e incorrecto ───────────────────
    {'nombre': 'Hablando Huevadas',                'categoria': 'humor_streaming',      'registro_linguistico': 'incorrecto', 'url': 'https://www.youtube.com/@HablandoHuevadasOficial',        'nota': 'humor adulto; groserías, peruanismos; referente para lenguaje ofensivo'},
    {'nombre': 'Todo Good',                        'categoria': 'streaming_opinion',    'registro_linguistico': 'coloquial',  'url': 'https://www.youtube.com/@todogoodpe',                     'nota': 'conversación, humor, invitados y coyuntura; mezcla registros'},
    {'nombre': 'Goblinciano',                      'categoria': 'streaming_opinion',    'registro_linguistico': 'coloquial',  'url': 'https://www.youtube.com/@Goblinciano',                    'nota': 'streaming largo; memes, opinión peruana, lenguaje muy coloquial'},
    {'nombre': 'El Cacas',                         'categoria': 'humor_streaming',      'registro_linguistico': 'coloquial',  'url': 'https://www.youtube.com/@ElCacas',                        'nota': 'humor y retos; jerga peruana urbana contemporánea'},
    {'nombre': 'Negro Fuertes',                    'categoria': 'humor_comedia',        'registro_linguistico': 'incorrecto', 'url': 'https://www.youtube.com/@NegroFuertes',                   'nota': 'comedia adulta; lenguaje soez y coloquial; referente de humor peruano popular'},
    {'nombre': 'Jason Qqq',                        'categoria': 'humor_streaming',      'registro_linguistico': 'incorrecto', 'url': 'https://www.youtube.com/@JasonQqqOficial',                'nota': 'reacciones y gaming; lenguaje muy informal y groserías frecuentes'},
    {'nombre': 'La Cotorrisa Peru',                'categoria': 'humor_podcast',        'registro_linguistico': 'coloquial',  'url': 'https://www.youtube.com/@LaCotorrisaPeru',                'nota': 'podcast de humor; adaptación peruana; lenguaje coloquial'},

    # ── Farándula / espectáculos ───────────────────────────────────────────────
    {'nombre': 'Magaly TV La Firme',               'categoria': 'farandula',            'registro_linguistico': 'informal',   'url': 'https://www.youtube.com/@MagalyTVLaFirmeATV',            'nota': 'farándula; conflicto público y lenguaje de espectáculos'},
    {'nombre': 'Amor y Fuego',                     'categoria': 'farandula',            'registro_linguistico': 'informal',   'url': 'https://www.youtube.com/@AmoryFuego',                    'nota': 'farándula y comentarios de entretenimiento'},
    {'nombre': 'America Hoy',                      'categoria': 'farandula',            'registro_linguistico': 'informal',   'url': 'https://www.youtube.com/@americahoytv',                  'nota': 'magazine y entretenimiento; verificar disponibilidad de videos'},
    {'nombre': 'Instarandula',                     'categoria': 'farandula_digital',    'registro_linguistico': 'coloquial',  'url': 'https://www.youtube.com/@Instarandula',                  'nota': 'Samuel Suárez; farándula digital; lenguaje coloquial con frases de peruanismos'},
    {'nombre': 'El Popular',                       'categoria': 'farandula_digital',    'registro_linguistico': 'informal',   'url': 'https://www.youtube.com/@ElPopularPeru',                 'nota': 'tabloid popular; espectáculos y farándula con lenguaje accesible'},

    # ── Deportes ──────────────────────────────────────────────────────────────
    {'nombre': 'Nico Moschella',                   'categoria': 'deportes_informal',    'registro_linguistico': 'incorrecto', 'url': 'https://www.youtube.com/@NicoMoschella',                 'nota': 'comentarista deportivo; lenguaje muy informal, apodos y jerga futbolera'},
    {'nombre': 'Líbero Deportes',                  'categoria': 'deportes',             'registro_linguistico': 'informal',   'url': 'https://www.youtube.com/@DiarioLiberoOficial',           'nota': 'diario Líbero; cobertura deportiva con lenguaje popular'},
    {'nombre': 'Depor',                            'categoria': 'deportes',             'registro_linguistico': 'informal',   'url': 'https://www.youtube.com/@DeporPeru',                     'nota': 'portal deportivo El Comercio; cobertura variada'},

    # ── Viajes / gastronomía ──────────────────────────────────────────────────
    {'nombre': 'Misias pero viajeras',             'categoria': 'viajes',               'registro_linguistico': 'coloquial',  'url': 'https://www.youtube.com/c/Misiasperoviajeras',           'nota': 'archivo de viajes; canal inactivo pero episodios útiles'},
    {'nombre': 'Buen Viaje',                       'categoria': 'viajes',               'registro_linguistico': 'informal',   'url': 'https://www.youtube.com/c/BuenViajePe',                 'nota': 'viajes por Perú con lenguaje descriptivo'},
    {'nombre': 'Viaja y Prueba',                   'categoria': 'viajes_gastronomia',   'registro_linguistico': 'informal',   'url': 'https://www.youtube.com/@ViajayPrueba',                 'nota': 'viajes y gastronomía por Perú'},
    {'nombre': 'Cocinando con la Patty',           'categoria': 'gastronomia',          'registro_linguistico': 'coloquial',  'url': 'https://www.youtube.com/@CocinandoConLaPatty',          'nota': 'cocina peruana; lenguaje cotidiano regional'},
])

# ── Canales aprobados de la corrida de scraping del 2026-07-18 ─────────────
# Criterios aplicados: canal peruano y oficial, actividad reciente o archivo
# pertinente, 3 videos inspeccionados y 3/3 con subtítulos automáticos en es.
balance_antes = canales_semilla['categoria'].value_counts().sort_index()
canales_nuevos_validados = pd.DataFrame([
    {'nombre': 'Arde Troya con Juliana Oxenford', 'categoria': 'politica_analisis', 'registro_linguistico': 'informal', 'url': 'https://www.youtube.com/@ardetroyalr', 'nota': 'análisis político y entrevistas; tono confrontacional', 'channel_id': 'UCgEabnv1xth8-qeWqMsTmDw'},
    {'nombre': 'Panorama', 'categoria': 'politica_periodismo', 'registro_linguistico': 'formal', 'url': 'https://www.youtube.com/@PanoramaPTV', 'nota': 'reportajes de investigación y actualidad peruana', 'channel_id': 'UCBiq92lt_ufNO-ktZigVXgg'},
    {'nombre': 'Juanito y Richard', 'categoria': 'humor_comedia', 'registro_linguistico': 'coloquial', 'url': 'https://www.youtube.com/@JuanitoyRichard', 'nota': 'comedia peruana, sketches y habla popular', 'channel_id': 'UCfiBnBtw8iNbX8vL4f6cp0Q'},
    {'nombre': 'Nada Espacial', 'categoria': 'humor_podcast', 'registro_linguistico': 'coloquial', 'url': 'https://www.youtube.com/@nadaespacialpodcast', 'nota': 'podcast peruano de humor, actualidad y conversación', 'channel_id': 'UCfwkQ4lY6UO-6K_REQ8CnNQ'},
    {'nombre': 'L1MAX', 'categoria': 'deportes_informal', 'registro_linguistico': 'informal', 'url': 'https://www.youtube.com/@L1MAX_', 'nota': 'canal oficial de Liga1; comentarios, previas y resúmenes del fútbol peruano', 'channel_id': 'UCGVHVLD7Nzw0zdIwbVhE3vw'},
    {'nombre': 'Cocina Cajamarquina', 'categoria': 'gastronomia', 'registro_linguistico': 'coloquial', 'url': 'https://www.youtube.com/@cocinacajamarquina', 'nota': 'recetas regionales y lenguaje cotidiano cajamarquino', 'channel_id': 'UC2lyekNE9NOGWu1R0L8046A'},
    {'nombre': 'Tío Lenguado y Descocaos', 'categoria': 'viajes_gastronomia', 'registro_linguistico': 'coloquial', 'url': 'https://www.youtube.com/@tiolenguado', 'nota': 'gastronomía, mercados y viajes con conversación familiar', 'channel_id': 'UCvPVK5xvxJRY1obR9XRrMfw'},
])
canales_nuevos_validados = canales_nuevos_validados.assign(
    origen='scraping_validado_2026-07-18',
    fecha_validacion='2026-07-18',
    videos_revisados=3,
    videos_con_subtitulos_es=3,
)

canales_semilla = canales_semilla.assign(
    channel_id=None,
    origen='semilla_preexistente',
    fecha_validacion='',
    videos_revisados=pd.NA,
    videos_con_subtitulos_es=pd.NA,
)
SEMILLAS_REVALIDADAS = {
    'https://www.youtube.com/@canalYAAAAA': 'UCP0AJJeNkFBYzegTTVbKhPg',
    'https://www.youtube.com/@exitosape': 'UCxgO_rak_BKZP8VNVmYqbWg',
    'https://www.youtube.com/@Panamericana-Noticias': 'UCOyD-kV3zB8Cm4LI4qtd9tA',
    'https://www.youtube.com/@MagalyTVLaFirmeATV': 'UCF6s3gpbZEQKqZ38VvQ0gLA',
    'https://www.youtube.com/@DiarioLiberoOficial': 'UCk2OZrA0E6q6xp4bBKtf9KA',
}
for url_validada, channel_id_validado in SEMILLAS_REVALIDADAS.items():
    mascara = canales_semilla['url'].eq(url_validada)
    canales_semilla.loc[mascara, [
        'channel_id', 'origen', 'fecha_validacion',
        'videos_revisados', 'videos_con_subtitulos_es',
    ]] = [
        channel_id_validado, 'semilla_revalidada_2026-07-18', '2026-07-18', 3, 3,
    ]
canales_semilla = pd.concat([canales_semilla, canales_nuevos_validados], ignore_index=True)


# ── Validaciones estructurales antes de usar o exportar las semillas ─────────
REGISTROS_VALIDOS = {'formal', 'informal', 'coloquial', 'incorrecto'}
CAMPOS_CANAL_REQUERIDOS = {'nombre', 'categoria', 'registro_linguistico', 'url', 'nota'}


def normalizar_url_validacion(url):
    return str(url or '').split('?', 1)[0].split('#', 1)[0].rstrip('/').casefold()


def validar_canales_semilla(df):
    errores = []
    faltantes = CAMPOS_CANAL_REQUERIDOS - set(df.columns)
    if faltantes:
        errores.append(f'columnas faltantes: {sorted(faltantes)}')
    for campo in CAMPOS_CANAL_REQUERIDOS & set(df.columns):
        vacios = df[campo].fillna('').astype(str).str.strip().eq('')
        if vacios.any():
            errores.append(f'{campo} vacío en filas {df.index[vacios].tolist()}')
    invalidos = sorted(set(df['registro_linguistico']) - REGISTROS_VALIDOS)
    if invalidos:
        errores.append(f'registros lingüísticos inválidos: {invalidos}')
    urls_norm = df['url'].map(normalizar_url_validacion)
    if urls_norm.duplicated().any():
        errores.append(f'URLs duplicadas: {df.loc[urls_norm.duplicated(False), "url"].tolist()}')
    patron_url = r'^https://www\.youtube\.com/(?:@|c/|channel/|user/)'
    urls_invalidas = ~df['url'].astype(str).str.match(patron_url, case=False, na=False)
    if urls_invalidas.any():
        errores.append(f'URLs de canal inválidas: {df.loc[urls_invalidas, "url"].tolist()}')
    revisados = df['origen'].isin({
        'scraping_validado_2026-07-18', 'semilla_revalidada_2026-07-18',
    })
    n_revisados = pd.to_numeric(df['videos_revisados'], errors='coerce').fillna(0)
    n_con_subs = pd.to_numeric(df['videos_con_subtitulos_es'], errors='coerce').fillna(0)
    evidencia_incompleta = revisados & (
        df['channel_id'].fillna('').eq('')
        | n_revisados.lt(3)
        | n_con_subs.lt(3)
    )
    if evidencia_incompleta.any():
        errores.append(f'evidencia incompleta en filas {df.index[evidencia_incompleta].tolist()}')
    if errores:
        raise ValueError('Canales semilla inválidos:\n- ' + '\n- '.join(errores))
    return pd.DataFrame({
        'validacion': ['estructura', 'registros', 'URLs únicas', 'evidencia nuevos'],
        'estado': ['OK', 'OK', 'OK', 'OK'],
    })


reporte_validacion_canales = validar_canales_semilla(canales_semilla)

# Si está disponible la última corrida, confirmar que los aprobados provienen de ella.
archivo_ultima_corrida = RAW_DIR / 'canales_candidatos_scraping.csv'
if archivo_ultima_corrida.exists():
    ultima_corrida = pd.read_csv(archivo_ultima_corrida)
    urls_corrida = set(ultima_corrida['url'].map(normalizar_url_validacion))
    urls_aprobadas = set(canales_nuevos_validados['url'].map(normalizar_url_validacion))
    no_encontradas = sorted(urls_aprobadas - urls_corrida)
    if no_encontradas:
        raise ValueError(f'Canales aprobados ausentes de la última corrida: {no_encontradas}')

balance_despues = canales_semilla['categoria'].value_counts().sort_index()
comparacion_balance = pd.concat(
    [balance_antes.rename('antes'), balance_despues.rename('despues')], axis=1
).fillna(0).astype(int)
comparacion_balance['agregados'] = comparacion_balance['despues'] - comparacion_balance['antes']
if comparacion_balance['despues'].min() < 2:
    raise ValueError('El balance objetivo exige al menos 2 canales por categoría.')

candidatos_por_verificar = pd.DataFrame([
    {'nombre': 'Phillip Butters',        'categoria': 'politica_opinion',  'consulta': 'Phillip Butters YouTube Peru canal oficial',        'nota': 'opinión política polémica; lenguaje directo y confrontacional; verificar canal activo'},
    {'nombre': 'Beto Ortiz',             'categoria': 'periodismo_opinion', 'consulta': 'Beto Ortiz YouTube Peru canal oficial entrevistas', 'nota': 'periodismo de investigación e entrevistas; verificar canal oficial'},
    {'nombre': 'Doble Merito',           'categoria': 'humor_streaming',    'consulta': 'Doble Merito YouTube Peru canal streaming',         'nota': 'humor peruano urbano; verificar URL y actividad'},
    {'nombre': 'Peluche Oficial',        'categoria': 'humor_comedia',      'consulta': 'Peluche youtuber peruano canal oficial',            'nota': 'comedia peruana; verificar canal y disponibilidad de subtítulos'},
    {'nombre': 'Renzo Reggiardo',        'categoria': 'politica_opinion',   'consulta': 'Renzo Reggiardo YouTube Peru canal oficial',        'nota': 'opinión y política; verificar si sube contenido regularmente'},
    {'nombre': 'DoblexD Peru',           'categoria': 'humor_gaming',       'consulta': 'DoblexD YouTube Peru gaming humor peruano',         'nota': 'gaming y humor; jerga gamer peruana; verificar URL'},
    {'nombre': 'Combate / EEG Peru',     'categoria': 'entretenimiento_tv', 'consulta': 'Combate EEG Peru YouTube canal oficial',            'nota': 'reality; lenguaje juvenil peruano; verificar canal actualizado'},
    {'nombre': 'La Paisana Jacinta',     'categoria': 'humor_regional',     'consulta': 'Paisana Jacinta YouTube Peru canal oficial',        'nota': 'humor regional peruano; acento y modismos andinos; verificar disponibilidad'},
])

canales_semilla.to_csv(RAW_DIR / 'canales_semilla.csv', index=False)
canales_nuevos_validados.to_csv(RAW_DIR / 'canales_nuevos_validados.csv', index=False)
reporte_validacion_canales.to_csv(RAW_DIR / 'validacion_canales_semilla.csv', index=False)
candidatos_por_verificar.to_csv(RAW_DIR / 'candidatos_por_verificar.csv', index=False)

print(f'Canales semilla      : {len(canales_semilla)}')
print(f'Nuevos validados     : {len(canales_nuevos_validados)}')
print(f'Candidatos a verificar: {len(candidatos_por_verificar)}')
print()
print('Balance por categoría (antes/después):')
display(comparacion_balance)
print('Validaciones:')
display(reporte_validacion_canales)


Canales semilla      : 36
Nuevos validados     : 7
Candidatos a verificar: 8

Balance por categoría (antes/después):


,antes,despues,agregados
categoria,,,
deportes,2,2,0
deportes_informal,1,2,1
farandula,3,3,0
farandula_digital,2,2,0
gastronomia,1,2,1
humor_comedia,1,2,1
humor_podcast,1,2,1
humor_streaming,3,3,0
politica_actualidad,6,6,0


Validaciones:


,validacion,estado
0,estructura,OK
1,registros,OK
2,URLs únicas,OK
3,evidencia nuevos,OK


## 4. Busqueda web con Selenium

Esta seccion reproduce la logica de scraping vista en clase: abrir navegador, cargar una pagina, obtener HTML y parsear enlaces. Si ya se tienen URLs verificadas, esta parte puede omitirse.

In [4]:
def iniciar_driver(headless=SELENIUM_HEADLESS):
    from selenium import webdriver
    from selenium.webdriver.chrome.service import Service
    from webdriver_manager.chrome import ChromeDriverManager

    options = webdriver.ChromeOptions()
    if headless:
        options.add_argument('--headless=new')
    options.add_argument('--disable-gpu')
    options.add_argument('--no-sandbox')
    options.add_argument('--disable-dev-shm-usage')
    options.add_argument(f'--window-size={NAVEGADOR_VENTANA}')
    options.add_argument(f'--lang={NAVEGADOR_IDIOMA}')
    options.add_argument(f'--user-agent={USER_AGENT}')
    # Ocultar señales de automatización
    options.add_experimental_option('excludeSwitches', ['enable-automation'])
    options.add_experimental_option('useAutomationExtension', False)
    service = Service(ChromeDriverManager().install())
    driver = webdriver.Chrome(service=service, options=options)
    # Eliminar propiedad webdriver del navegador
    driver.execute_cdp_cmd(
        'Page.addScriptToEvaluateOnNewDocument',
        {'source': 'Object.defineProperty(navigator, "webdriver", {get: () => undefined})'}
    )
    return driver


def buscar_canales_youtube(consulta, n_scrolls=SELENIUM_N_SCROLLS,
                             headless=SELENIUM_HEADLESS):
    from selenium.webdriver.common.by import By
    from selenium.webdriver.support.ui import WebDriverWait
    from selenium.webdriver.support import expected_conditions as EC

    driver = iniciar_driver(headless=headless)
    try:
        # &sp=EgIQAg%3D%3D filtra resultados por tipo 'Canal'
        url = ('https://www.youtube.com/results?search_query='
               + quote_plus(consulta) + '&sp=EgIQAg%3D%3D')
        driver.get(url)
        try:
            WebDriverWait(driver, SELENIUM_WAIT_SECONDS).until(
                EC.presence_of_element_located((By.TAG_NAME, 'ytd-channel-renderer'))
            )
        except Exception:
            time.sleep(SELENIUM_FALLBACK_SECONDS)
        for _ in range(n_scrolls):
            driver.execute_script('window.scrollTo(0, document.documentElement.scrollHeight);')
            time.sleep(SELENIUM_SCROLL_PAUSE_SECONDS)
        html = driver.page_source
    finally:
        driver.quit()

    soup = BeautifulSoup(html, 'html.parser')
    filas = []
    for a in soup.find_all('a', href=True):
        href = a['href']
        texto = a.get_text(' ', strip=True)
        if ('/@' in href or '/channel/' in href or '/c/' in href) and texto:
            if href.startswith('/'):
                href = 'https://www.youtube.com' + href
            filas.append({'consulta': consulta, 'nombre': texto, 'url': href.split('?')[0]})
    return pd.DataFrame(filas, columns=['consulta', 'nombre', 'url']).drop_duplicates('url')


# Las consultas editables están agrupadas en CONSULTAS_BUSQUEDA al inicio.


In [6]:
# Ejecutar esta celda si se desea descubrir nuevos canales desde la web publica.
# Puede tardar por la carga dinamica de YouTube.

if EJECUTAR_BUSQUEDA_CANALES:
    candidatos = []
    for consulta in tqdm(CONSULTAS_BUSQUEDA):
        candidatos.append(buscar_canales_youtube(consulta))
    canales_candidatos = pd.concat(candidatos, ignore_index=True).drop_duplicates('url')
    canales_candidatos.to_csv(RAW_DIR / 'canales_candidatos_scraping.csv', index=False)
else:
    canales_candidatos = canales_semilla.copy()

canales_candidatos.head(20)

,nombre,categoria,registro_linguistico,url,nota,channel_id,origen,fecha_validacion,videos_revisados,videos_con_subtitulos_es
0,Marco Sifuentes / Ocram,politica_analisis,informal,https://www.youtube.com/@canalYAAAAA,La Encerrona; análisis político con tono irónico,UCP0AJJeNkFBYzegTTVbKhPg,semilla_revalidada_2026-07-18,2026-07-18,3,3
1,El diario de Curwen,politica_opinion,coloquial,https://www.youtube.com/@curwen,opinión política con sarcasmo y lenguaje muy c...,None,semilla_preexistente,,<NA>,<NA>
2,Sin Guion con Rosa Maria Palacios,politica_periodismo,formal,https://www.youtube.com/@singuionlr,periodismo de opinión y entrevistas,None,semilla_preexistente,,<NA>,<NA>
3,RPP Noticias,politica_actualidad,formal,https://www.youtube.com/@RPPNoticias,noticias y entrevistas; lenguaje periodístico ...,None,semilla_preexistente,,<NA>,<NA>
4,Exitosa Noticias,politica_actualidad,informal,https://www.youtube.com/@exitosape,"noticias, entrevistas y opinión; tono más popu...",UCxgO_rak_BKZP8VNVmYqbWg,semilla_revalidada_2026-07-18,2026-07-18,3,3
5,Willax Television,politica_opinion,informal,https://www.youtube.com/@WillaxTV,programas de opinión política; lenguaje a vece...,None,semilla_preexistente,,<NA>,<NA>
6,Canal N,politica_actualidad,formal,https://www.youtube.com/@canaln,noticias y entrevistas; canal de cable informa...,None,semilla_preexistente,,<NA>,<NA>
7,ATV Noticias,politica_actualidad,informal,https://www.youtube.com/@ATVNoticias,noticias nacionales; lenguaje más popular que ...,None,semilla_preexistente,,<NA>,<NA>
8,Latina Noticias,politica_actualidad,informal,https://www.youtube.com/@latinanoticias,noticias y magazine; amplia cobertura nacional,None,semilla_preexistente,,<NA>,<NA>
9,Panamericana Noticias,politica_actualidad,informal,https://www.youtube.com/@Panamericana-Noticias,canal informativo oficial de Panamericana Tele...,UCOyD-kV3zB8Cm4LI4qtd9tA,semilla_revalidada_2026-07-18,2026-07-18,3,3


## 5. Extraccion de metadatos de videos con yt-dlp

`yt-dlp` permite leer metadatos publicos y subtitulos sin usar una API key. Para mantener el corpus controlado, se limita el numero de videos por canal.

In [7]:
import yt_dlp

YT_HEADERS = {'User-Agent': USER_AGENT}

# Opciones base: todas las operaciones de este cuaderno omiten el contenido
# audiovisual. `skip_download` es una salvaguarda global, no solo de subtítulos.
_YT_BASE_OPTS = {
    'quiet': YT_QUIET,
    'no_warnings': YT_NO_WARNINGS,
    'ignoreerrors': YT_IGNORE_ERRORS,
    'skip_download': True,
    'nocheckcertificate': PERMITIR_SSL_INSEGURO,
    'extractor_retries': YT_RETRIES,
    'http_headers': YT_HEADERS,
    'sleep_interval': YT_SLEEP_MIN_SECONDS,
    'max_sleep_interval': YT_SLEEP_MAX_SECONDS,
}


def _normalizar_url_canal(channel_url):
    """Garantiza que la URL apunte a la pestaña /videos del canal."""
    base = channel_url.rstrip('/')
    for sufijo in ('/videos', '/streams', '/shorts', '/playlists'):
        if base.endswith(sufijo):
            base = base[: -len(sufijo)]
    return base + '/videos'


def listar_videos_canal(channel_url, max_videos=MAX_VIDEOS_POR_CANAL):
    url_videos = _normalizar_url_canal(channel_url)
    opciones = {
        **_YT_BASE_OPTS,
        'extract_flat': 'in_playlist',
        'playlist_items': f'1:{max_videos}',
        'skip_download': True,
    }
    try:
        with yt_dlp.YoutubeDL(opciones) as ydl:
            info = ydl.extract_info(url_videos, download=False)
    except Exception as exc:
        print(f'  ✗ Error extrayendo canal {channel_url}: {exc}')
        return []

    if not info:
        return []

    filas = []
    for item in info.get('entries', []) or []:
        if not item:
            continue
        video_id = item.get('id')
        if not video_id:
            continue
        filas.append({
            'video_id': video_id,
            'url': f'https://www.youtube.com/watch?v={video_id}',
            'title': item.get('title'),
            'upload_date': item.get('upload_date'),
            'duration': item.get('duration'),
            'view_count': item.get('view_count'),
            'channel_id': item.get('channel_id'),
            'channel_url': channel_url,
        })
    return filas


In [8]:
videos = []
for _, canal in tqdm(canales_semilla.iterrows(), total=len(canales_semilla)):
    try:
        filas = listar_videos_canal(canal['url'], max_videos=MAX_VIDEOS_POR_CANAL)
        for fila in filas:
            fila['channel_title'] = canal['nombre']
            fila['categoria_fuente'] = canal['categoria']
        videos.extend(filas)
    except Exception as exc:
        print('No se pudo leer canal:', canal['nombre'], exc)

COLUMNAS_VIDEO = [
    'video_id', 'url', 'title', 'upload_date', 'duration', 'view_count',
    'channel_id', 'channel_url', 'channel_title', 'categoria_fuente',
]
videos_df = pd.DataFrame(videos, columns=COLUMNAS_VIDEO)
videos_df = videos_df.drop_duplicates('video_id').reset_index(drop=True)
videos_df.to_csv(RAW_DIR / 'videos_candidatos.csv', index=False)
videos_df.head()

  0%|          | 0/36 [00:00<?, ?it/s]

ERROR: [youtube:tab] @canaln: Unable to download API page: HTTP Error 404: Not Found (caused by <HTTPError 404: Not Found>)
ERROR: [youtube:tab] @ElCacas: This channel does not have a videos tab
ERROR: [youtube:tab] @NegroFuertes: Unable to download API page: HTTP Error 404: Not Found (caused by <HTTPError 404: Not Found>)
ERROR: [youtube:tab] @JasonQqqOficial: Unable to download API page: HTTP Error 404: Not Found (caused by <HTTPError 404: Not Found>)
ERROR: [youtube:tab] @LaCotorrisaPeru: Unable to download API page: HTTP Error 404: Not Found (caused by <HTTPError 404: Not Found>)
ERROR: [youtube:tab] @AmoryFuego: This channel does not have a videos tab
ERROR: [youtube:tab] @americahoytv: Unable to download API page: HTTP Error 404: Not Found (caused by <HTTPError 404: Not Found>)
ERROR: [youtube:tab] @NicoMoschella: Unable to download API page: HTTP Error 404: Not Found (caused by <HTTPError 404: Not Found>)
ERROR: [youtube:tab] @DeporPeru: Unable to download API page: HTTP Error 4

,video_id,url,title,upload_date,duration,view_count,channel_id,channel_url,channel_title,categoria_fuente
0,uh5PlvYoslk,https://www.youtube.com/watch?v=uh5PlvYoslk,NEW congresspeople secure jobs for 2031 #LaEnc...,None,1337.0,None,None,https://www.youtube.com/@canalYAAAAA,Marco Sifuentes / Ocram,politica_analisis
1,1KUfzZWBTm0,https://www.youtube.com/watch?v=1KUfzZWBTm0,"RENIEC Chief comes out: ""Keiko will be the wis...",None,1271.0,None,None,https://www.youtube.com/@canalYAAAAA,Marco Sifuentes / Ocram,politica_analisis
2,C7QbCoDkjeI,https://www.youtube.com/watch?v=C7QbCoDkjeI,Nieto asks Keiko for Pedro Castillo's FREEDOM ...,None,1409.0,None,None,https://www.youtube.com/@canalYAAAAA,Marco Sifuentes / Ocram,politica_analisis
3,OsG4Iie8G6I,https://www.youtube.com/watch?v=OsG4Iie8G6I,MARINE ANIMALS DYING DUE TO OCEAN WARMING | Oc...,None,1230.0,None,None,https://www.youtube.com/@canalYAAAAA,Marco Sifuentes / Ocram,politica_analisis
4,TTIGd1fZQA0,https://www.youtube.com/watch?v=TTIGd1fZQA0,Rospigliosi to be sued for DEFAMATION #LaEncer...,None,1410.0,None,None,https://www.youtube.com/@canalYAAAAA,Marco Sifuentes / Ocram,politica_analisis


## 6. Descarga de subtitulos publicos

Se descargan subtitulos en espanol cuando existan. Los registros y archivos VTT ya presentes se validan antes de reutilizarlos: si estan integros no se vuelven a solicitar; solo se reintentan los vacios, truncados o incompatibles. Si un video no tiene subtitulos, queda fuera del corpus inicial.

In [9]:
SUBS_DIR = RAW_DIR / 'subtitulos'
SUBS_DIR.mkdir(parents=True, exist_ok=True)


def archivos_vtt(video_id):
    """Lista VTT del video priorizando español peruano y latinoamericano."""
    prioridad = {idioma: indice for indice, idioma in enumerate(IDIOMAS_SUBTITULOS)}

    def clave(path):
        partes = path.name.split('.')
        idioma = partes[-2] if len(partes) >= 3 else ''
        return (prioridad.get(idioma, 99), path.name)

    return sorted(SUBS_DIR.glob(f'{video_id}*.vtt'), key=clave)


def descargar_subtitulos(video_url, video_id):
    """Descarga solo VTT en español; nunca descarga audio ni video.

    yt-dlp prioriza subtítulos manuales y usa automáticos como respaldo cuando
    se habilitan simultáneamente writesubtitles y writeautomaticsub.
    """
    outtmpl = str(SUBS_DIR / f'{video_id}.%(ext)s')
    opciones = {
        **_YT_BASE_OPTS,
        'noplaylist': True,
        'writesubtitles': OBTENER_SUBTITULOS_MANUALES,
        'writeautomaticsub': OBTENER_SUBTITULOS_AUTOMATICOS,
        'subtitleslangs': IDIOMAS_SUBTITULOS,
        'subtitlesformat': FORMATO_SUBTITULOS,
        'outtmpl': outtmpl,
    }
    try:
        with yt_dlp.YoutubeDL(opciones) as ydl:
            ydl.download([video_url])
    except Exception as exc:
        tqdm.write(f'  yt-dlp no pudo obtener subtítulos de {video_id}: {exc}')
    archivos = archivos_vtt(video_id)
    return archivos[0] if archivos else None


def descargar_subtitulos_transcript_api(video_id):
    """Fallback sin API key usando la interfaz vigente (>= 1.2)."""
    try:
        from requests import Session
        from youtube_transcript_api import YouTubeTranscriptApi, NoTranscriptFound

        sesion = Session()
        sesion.verify = not PERMITIR_SSL_INSEGURO
        ytt_api = YouTubeTranscriptApi(http_client=sesion)
        transcript_list = ytt_api.list(video_id)
        try:
            transcript = transcript_list.find_manually_created_transcript(IDIOMAS_SUBTITULOS)
        except NoTranscriptFound:
            transcript = transcript_list.find_generated_transcript(IDIOMAS_SUBTITULOS)
        return transcript.fetch().to_raw_data()
    except Exception as exc:
        tqdm.write(f'  transcript-api no disponible para {video_id}: {exc}')
        return []


def tiempo_a_segundos(valor):
    partes = valor.replace(',', '.').split(':')
    partes = [float(p) for p in partes]
    if len(partes) == 3:
        h, m, s = partes
    else:
        h, m, s = 0, partes[0], partes[1]
    return h * 3600 + m * 60 + s


def limpiar_linea_vtt(linea):
    linea = re.sub(r'<[^>]+>', '', linea)
    linea = re.sub(r'&amp;', '&', linea)
    linea = re.sub(r'&nbsp;', ' ', linea)
    linea = re.sub(r'\s+', ' ', linea).strip()
    return linea


def leer_vtt(path):
    texto = path.read_text(encoding='utf-8', errors='ignore')
    bloques = re.split(r'\n\s*\n', texto)
    segmentos = []
    vistos = set()
    patron_tiempo = re.compile(
        r'(\d{2}:\d{2}:\d{2}\.\d{3}|\d{2}:\d{2}\.\d{3})'
        r'\s+-->\s+'
        r'(\d{2}:\d{2}:\d{2}\.\d{3}|\d{2}:\d{2}\.\d{3})'
    )
    for bloque in bloques:
        lineas = [ln.strip() for ln in bloque.splitlines() if ln.strip()]
        if not lineas:
            continue
        match = None
        idx = 0
        for i, linea in enumerate(lineas):
            match = patron_tiempo.search(linea)
            if match:
                idx = i
                break
        if not match:
            continue
        start = tiempo_a_segundos(match.group(1))
        end = tiempo_a_segundos(match.group(2))
        frase = ' '.join(limpiar_linea_vtt(ln) for ln in lineas[idx + 1:])
        frase = re.sub(r'\s+', ' ', frase).strip()
        clave = (round(start, 1), frase.lower())
        if frase and clave not in vistos:
            vistos.add(clave)
            segmentos.append({'start': start, 'duration': max(end - start, 0.1), 'text': frase})
    return segmentos


In [14]:
def hash_texto(texto):
    return hashlib.md5(texto.encode('utf-8')).hexdigest()


CAMPOS_REQUERIDOS_NB02 = {'video_id', 'title', 'channel_title', 'segments'}


def valor_json(value):
    """Convierte valores nulos de pandas/numpy a None para JSON estándar."""
    try:
        return None if pd.isna(value) else value
    except (TypeError, ValueError):
        return value


def normalizar_segmentos(segmentos):
    salida_segmentos = []
    for seg in segmentos or []:
        try:
            texto = str(seg.get('text') or '').strip()
            inicio = float(seg.get('start', 0.0))
            duracion = max(float(seg.get('duration', 0.0)), 0.0)
        except (AttributeError, TypeError, ValueError):
            continue
        if texto and math.isfinite(inicio) and math.isfinite(duracion):
            salida_segmentos.append({
                'start': inicio, 'duration': duracion, 'text': texto,
            })
    return salida_segmentos


def texto_segmentos(segmentos):
    return ' '.join(seg['text'] for seg in segmentos).strip()


def transcripcion_integra(record):
    """Valida el contrato de NB02 antes de considerar un video procesado."""
    if not isinstance(record, dict) or CAMPOS_REQUERIDOS_NB02 - record.keys():
        return False
    if not record.get('video_id'):
        return False
    segmentos = normalizar_segmentos(record.get('segments'))
    return len(texto_segmentos(segmentos)) >= MIN_TRANSCRIPT_CHARS


def vtt_integro(segmentos):
    """Un VTT es reutilizable si produjo segmentos válidos y texto suficiente."""
    return len(texto_segmentos(normalizar_segmentos(segmentos))) >= MIN_TRANSCRIPT_CHARS


def mejor_vtt(video_id):
    """Devuelve (segmentos, path) del VTT más completo disponible."""
    candidatos = []
    for path in archivos_vtt(video_id):
        segmentos = normalizar_segmentos(leer_vtt(path))
        candidatos.append((len(texto_segmentos(segmentos)), segmentos, path))
    if not candidatos:
        return [], None
    _, segmentos, path = max(candidatos, key=lambda item: item[0])
    return segmentos, path


# Cargar registros válidos ya guardados para que la ejecución sea reanudable.
salida = RAW_DIR / 'transcripts_raw.jsonl'
transcripciones_por_id = {}
if REUTILIZAR_CACHE and salida.exists():
    with open(salida, encoding='utf-8') as f:
        for numero_linea, linea in enumerate(f, start=1):
            try:
                row = json.loads(linea)
                row['segments'] = normalizar_segmentos(row.get('segments'))
                if not transcripcion_integra(row):
                    raise ValueError('registro incompleto, corrupto o incompatible con NB02')
                transcripciones_por_id[row['video_id']] = row
            except (json.JSONDecodeError, TypeError, ValueError) as exc:
                print(f'Advertencia: línea {numero_linea} omitida en {salida.name}: {exc}')

ids_ya_procesados = set(transcripciones_por_id)
print(f'Transcripciones previas válidas: {len(ids_ya_procesados)}')

sin_subs = []
pendientes = videos_df[~videos_df['video_id'].isin(ids_ya_procesados)]
print(f'Videos pendientes de procesar   : {len(pendientes)} / {len(videos_df)}')

barra = tqdm(pendientes.iterrows(), total=len(pendientes), desc='Subtítulos', unit='video')
for _, video in barra:
    vid = video['video_id']
    segmentos = []
    fuente_subs = None

    # ── Reutilizar el mejor VTT en caché ────────────────────────────────────
    segmentos, vtt_path = mejor_vtt(vid) if REUTILIZAR_CACHE else ([], None)
    if vtt_integro(segmentos):
        fuente_subs = 'yt-dlp-vtt (cache)'
    else:
        # Intento 1: yt-dlp escribe únicamente archivos de subtítulos.
        try:
            descargar_subtitulos(video['url'], vid)
            segmentos, vtt_path = mejor_vtt(vid)
            if vtt_integro(segmentos):
                fuente_subs = 'yt-dlp-vtt'
        except Exception as exc:
            tqdm.write(f'  yt-dlp falló en {vid}: {exc}')

        # Intento 2: transcripción en memoria; tampoco descarga el video.
        if USAR_TRANSCRIPT_API_FALLBACK and not vtt_integro(segmentos):
            segmentos_api = normalizar_segmentos(descargar_subtitulos_transcript_api(vid))
            if vtt_integro(segmentos_api):
                segmentos = segmentos_api
                fuente_subs = 'transcript-api'

    texto = ' '.join(seg['text'] for seg in segmentos)
    if len(texto) < MIN_TRANSCRIPT_CHARS:
        sin_subs.append(vid)
        barra.set_postfix(ok=len(transcripciones_por_id), sin_subs=len(sin_subs), fuente='—')
        continue

    transcripciones_por_id[vid] = {
        'video_id': vid,
        'url': valor_json(video.get('url')),
        'title': valor_json(video.get('title')),
        'channel_id': valor_json(video.get('channel_id')),
        'channel_title': valor_json(video.get('channel_title')),
        'channel_url': valor_json(video.get('channel_url')),
        'published_at': valor_json(video.get('upload_date')),
        'categoria_fuente': valor_json(video.get('categoria_fuente')),
        'fuente_subs': fuente_subs,
        'text_hash': hash_texto(texto),
        'segments': segmentos,
    }
    barra.set_postfix(ok=len(transcripciones_por_id), sin_subs=len(sin_subs), fuente=fuente_subs or '—')

# Escritura atómica: si la sesión se interrumpe, se conserva el JSONL anterior.
transcripciones = list(transcripciones_por_id.values())
salida_temporal = salida.with_suffix('.jsonl.tmp')
with open(salida_temporal, 'w', encoding='utf-8') as f:
    for row in transcripciones:
        faltantes = CAMPOS_REQUERIDOS_NB02 - row.keys()
        if faltantes:
            raise ValueError(f'Registro {row.get("video_id")} incompatible con NB02: {faltantes}')
        f.write(json.dumps(row, ensure_ascii=False, allow_nan=False) + '\n')
salida_temporal.replace(salida)

print(f'\nTranscripciones totales           : {len(transcripciones)}')
print(f'  - previas válidas               : {len(ids_ya_procesados)}')
print(f'  - nuevas en esta ejecución      : {len(set(transcripciones_por_id) - ids_ya_procesados)}')
print(f'Sin subtítulos (omitidos)         : {len(sin_subs)}')
n_cubiertos = videos_df['video_id'].isin(transcripciones_por_id).sum()
tasa = n_cubiertos / max(len(videos_df), 1) * 100
print(f'Tasa de cobertura                 : {tasa:.1f}%')
print(f'Esquema compatible con NB02       : {sorted(CAMPOS_REQUERIDOS_NB02)}')
print(f'Archivo: {salida}')


Transcripciones previas válidas: 1847
Videos pendientes de procesar   : 71 / 1817


Subtítulos:   0%|          | 0/71 [00:00<?, ?video/s]

ERROR: [youtube] LFKGSJKX7TY: Join this channel to get access to members-only content like this video, and other exclusive perks.


  transcript-api no disponible para LFKGSJKX7TY: 
Could not retrieve a transcript for the video https://www.youtube.com/watch?v=LFKGSJKX7TY! This is most likely caused by:

The video is unplayable for the following reason: Join this channel to get access to members-only content like this video, and other exclusive perks.

If you are sure that the described cause is not responsible for this error and that a transcript should be retrievable, please create an issue at https://github.com/jdepoix/youtube-transcript-api/issues. Please add which version of youtube_transcript_api you are using and provide the information needed to replicate the error. Also make sure that there are no open issues which already describe your problem!


ERROR: [youtube] AumPu5lPMp8: Join this channel to get access to members-only content like this video, and other exclusive perks.


...

  transcript-api no disponible para tO8V-XEM2v0: 
Could not retrieve a transcript for the video https://www.youtube.com/watch?v=tO8V-XEM2v0! This is most likely caused by:

This video is age-restricted. Therefore, you are unable to retrieve transcripts for it without authenticating yourself.

Unfortunately, Cookie Authentication is temporarily unsupported in youtube-transcript-api, as recent changes in YouTube's API broke the previous implementation. I will do my best to re-implement it as soon as possible.

If you are sure that the described cause is not responsible for this error and that a transcript should be retrievable, please create an issue at https://github.com/jdepoix/youtube-transcript-api/issues. Please add which version of youtube_transcript_api you are using and provide the information needed to replicate the error. Also make sure that there are no open issues which already describe your problem!
  transcript-api no disponible para iXto1fh1a60: 
Could not retrieve a trans

ERROR: [youtube] GM3QdzndLLw: This video is not available
ERROR: [youtube] 8Xlz2EelLDA: This video is not available
ERROR: [youtube] drgLilD2bnA: This video is not available
ERROR: [youtube] CWyBtgzUlqg: This video is not available
ERROR: [youtube] 1MPPLtM6BA4: This video is not available


  transcript-api no disponible para SDAMs7BqLVs:      
Could not retrieve a transcript for the video https://www.youtube.com/watch?v=SDAMs7BqLVs! This is most likely caused by:

No transcripts were found for any of the requested language codes: ['es-PE', 'es-419', 'es']

For this video (SDAMs7BqLVs) transcripts are available in the following languages:

(MANUALLY CREATED)
None

(GENERATED)
 - en ("English (auto-generated)")[TRANSLATABLE]

(TRANSLATION LANGUAGES)
 - ar ("Arabic")
 - zh-Hant ("Chinese (Traditional)")
 - nl ("Dutch")
 - fr ("French")
 - de ("German")
 - hi ("Hindi")
 - id ("Indonesian")
 - it ("Italian")
 - ja ("Japanese")
 - ko ("Korean")
 - pt ("Portuguese")
 - ru ("Russian")
 - es ("Spanish")
 - th ("Thai")
 - uk ("Ukrainian")
 - vi ("Vietnamese")

If you are sure that the described cause is not responsible for this error and that a transcript should be retrievable, please create an issue at https://github.com/jdepoix/youtube-transcript-api/issues. Please add which ver

ERROR: [youtube] l8KQ0ShrDwo: Sign in to confirm your age. This video may be inappropriate for some users. Use --cookies-from-browser or --cookies for the authentication. See  https://github.com/yt-dlp/yt-dlp/wiki/FAQ#how-do-i-pass-cookies-to-yt-dlp  for how to manually pass cookies. Also see  https://github.com/yt-dlp/yt-dlp/wiki/Extractors#exporting-youtube-cookies  for tips on effectively exporting YouTube cookies


  transcript-api no disponible para l8KQ0ShrDwo: 
Could not retrieve a transcript for the video https://www.youtube.com/watch?v=l8KQ0ShrDwo! This is most likely caused by:

This video is age-restricted. Therefore, you are unable to retrieve transcripts for it without authenticating yourself.

Unfortunately, Cookie Authentication is temporarily unsupported in youtube-transcript-api, as recent changes in YouTube's API broke the previous implementation. I will do my best to re-implement it as soon as possible.

If you are sure that the described cause is not responsible for this error and that a transcript should be retrievable, please create an issue at https://github.com/jdepoix/youtube-transcript-api/issues. Please add which version of youtube_transcript_api you are using and provide the information needed to replicate the error. Also make sure that there are no open issues which already describe your problem!
  transcript-api no disponible para 6pOex1veqwQ: 
Could not retrieve a trans

## 7. Verificacion: solo subtitulos

Este cuaderno no descarga audio ni video. La celda siguiente comprueba las salvaguardas y resume los archivos de subtítulos generados.

In [15]:
assert _YT_BASE_OPTS['skip_download'] is True
assert 'format' not in _YT_BASE_OPTS

extensiones_multimedia = {'.mp4', '.webm', '.mkv', '.m4a', '.mp3', '.opus'}
archivos_generados = [p for p in SUBS_DIR.iterdir() if p.is_file()]
multimedia_en_subtitulos = [p for p in archivos_generados if p.suffix.lower() in extensiones_multimedia]
if multimedia_en_subtitulos:
    raise RuntimeError(f'Se encontraron archivos multimedia inesperados en {SUBS_DIR}: {multimedia_en_subtitulos[:3]}')

print('Salvaguarda skip_download:', _YT_BASE_OPTS['skip_download'])
print('Archivos VTT disponibles :', len(list(SUBS_DIR.glob('*.vtt'))))
print('Audio/video descargado por este flujo: NO')


Salvaguarda skip_download: True
Archivos VTT disponibles : 1857
Audio/video descargado por este flujo: NO


## 8. Revision rapida

Antes de pasar al cuaderno 02, se recomienda revisar manualmente canales y transcripciones para retirar subtítulos deficientes, publicidad extensa o segmentos que no pertenecen al objetivo del corpus.

In [16]:
resumen = pd.DataFrame([
    {'archivo': 'canales_semilla.csv', 'ruta': str(RAW_DIR / 'canales_semilla.csv')},
    {'archivo': 'videos_candidatos.csv', 'ruta': str(RAW_DIR / 'videos_candidatos.csv')},
    {'archivo': 'transcripts_raw.jsonl', 'ruta': str(RAW_DIR / 'transcripts_raw.jsonl')},
])
resumen

,archivo,ruta
0,canales_semilla.csv,D:\trabajo_PLN\Trabajo_PLN-MIA-Grupo4\datos\ra...
1,videos_candidatos.csv,D:\trabajo_PLN\Trabajo_PLN-MIA-Grupo4\datos\ra...
2,transcripts_raw.jsonl,D:\trabajo_PLN\Trabajo_PLN-MIA-Grupo4\datos\ra...


In [17]:
# ── Estadísticas finales: videos listados × subtítulos utilizables ────────────
ids_con_transcript = {t['video_id'] for t in transcripciones}

stat = (
    videos_df
    .assign(
        transcript = videos_df['video_id'].isin(ids_con_transcript),
    )
    .groupby('channel_title', sort=False)
    .agg(
        listados   = ('video_id',   'count'),
        subtitulos = ('transcript', 'sum'),
    )
    .reset_index()
    .sort_values('subtitulos', ascending=False)
    .rename(columns={'channel_title': 'canal'})
)

stat['% subtitulos'] = (stat['subtitulos'] / stat['listados'] * 100).round(1)

# ── Totales ───────────────────────────────────────────────────────────────────
tot_l = stat['listados'].sum()
tot_s = stat['subtitulos'].sum()

COL = 32
print(f"\n{'Canal':<{COL}} {'Listados':>8}  {'Subtítulos':>11} {'%':>6}")
print('─' * (COL + 30))
for _, r in stat.iterrows():
    print(f"{r['canal'][:COL-1]:<{COL}} {r['listados']:>8}  "
          f"{r['subtitulos']:>11} {r['% subtitulos']:>5.1f}%")
print('─' * (COL + 30))
print(f"{'TOTAL':<{COL}} {tot_l:>8}  "
      f"{tot_s:>11} {tot_s/max(tot_l,1)*100:>5.1f}%")

# ── Tabla interactiva ─────────────────────────────────────────────────────────
stat.style.bar(subset=['% subtitulos'], color='#5fba7d', vmin=0, vmax=100)



Canal                            Listados   Subtítulos      %
──────────────────────────────────────────────────────────────
Marco Sifuentes / Ocram                75           75 100.0%
Sin Guion con Rosa Maria Palaci        75           75 100.0%
Latina Noticias                        75           75 100.0%
RPP Noticias                           75           75 100.0%
Exitosa Noticias                       75           75 100.0%
Willax Television                      75           75 100.0%
Panamericana Noticias                  75           75 100.0%
Misias pero viajeras                   75           75 100.0%
Buen Viaje                             75           75 100.0%
Magaly TV La Firme                     75           75 100.0%
Tío Lenguado y Descocaos               75           75 100.0%
Panorama                               75           75 100.0%
Arde Troya con Juliana Oxenford        75           75 100.0%
Líbero Deportes                        75           75 100.0%
Viaja 

,canal,listados,subtitulos,% subtitulos
0,Marco Sifuentes / Ocram,75,75,100.000000
2,Sin Guion con Rosa Maria Palacios,75,75,100.000000
7,Latina Noticias,75,75,100.000000
3,RPP Noticias,75,75,100.000000
4,Exitosa Noticias,75,75,100.000000
5,Willax Television,75,75,100.000000
8,Panamericana Noticias,75,75,100.000000
16,Misias pero viajeras,75,75,100.000000
17,Buen Viaje,75,75,100.000000
12,Magaly TV La Firme,75,75,100.000000
